# FATE — Frequency-Augmented Transformer Encoder (Train)

A Transformer encoder that utilizes global character frequency as additional information.

**Usage:** Run the cells sequentially (Run All). In Colab, upload the repository folder to `MyDrive/cryptanalysis-ai`; locally, run the notebook from the `notebooks/` directory.

In [ ]:
# === SETUP (Colab or local) ===
import sys
from pathlib import Path

def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

if in_colab():
    from google.colab import drive
    drive.mount('/content/drive')
    # Upload the whole repo folder under MyDrive/cryptanalysis-ai
    # (or from a git repo: !git clone <URL> {REPO_DIR})
    REPO_DIR = Path('/content/drive/MyDrive/cryptanalysis-ai')
    %pip install -q pytorch-lightning torchmetrics tensorboard
else:
    # Local: this notebook is in notebooks/ -> repo root is one level up
    REPO_DIR = Path.cwd()
    if REPO_DIR.name == 'notebooks':
        REPO_DIR = REPO_DIR.parent

SRC = REPO_DIR / 'src'
assert SRC.exists(), f'src bulunamadi: {SRC}'
sys.path.insert(0, str(SRC))
print('Repo :', REPO_DIR)


In [ ]:
# === Config + imports ===
import config
import torch
import pytorch_lightning as pl
from pytorch_lightning.callbacks import (
    ModelCheckpoint, EarlyStopping, LearningRateMonitor, RichProgressBar
)
from pytorch_lightning.loggers import TensorBoardLogger

print(config.describe())


In [ ]:
# === Data ===
from models.substitution_common.data_module import SubstitutionDataModule

SEQ_LEN    = 256
BATCH_SIZE = 128

dm = SubstitutionDataModule(
    text_path=str(config.data_file("final_dataset_shuffled.txt")),
    alphabet=config.ALPHABET,
    seq_len=SEQ_LEN,
    batch_size=BATCH_SIZE,
    val_split=0.1,
    num_workers=2,
    seed=42,
)
dm.setup()
VOCAB_SIZE = dm.vocab_size 
print("vocab_size:", VOCAB_SIZE)


In [ ]:
# === Model ===
from models.substitution_fate.lightning import SubstitutionCipherSolverFreq

model = SubstitutionCipherSolverFreq(
    vocab_size=VOCAB_SIZE,
    embed_dim=256, num_heads=8, num_layers=6, ff_dim=1024,
    dropout=0.1, max_len=SEQ_LEN, warmup_steps=400,
)
n = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parametre: {n:,}  (~{n/1e6:.1f}M)")


In [ ]:
# === Trainer + callbacks ===
MODEL_NAME = "substitution_fate"
MAX_EPOCHS = 200
ckpt_dir, log_dir = config.model_paths(MODEL_NAME)

tb_logger = TensorBoardLogger(save_dir=str(log_dir), name=MODEL_NAME, log_graph=False)

best_ckpt = ModelCheckpoint(
    dirpath=str(ckpt_dir), filename="best-{epoch:02d}-{val_acc:.4f}",
    monitor="val_acc", mode="max", save_top_k=3, verbose=True,
)
last_ckpt = ModelCheckpoint(
    dirpath=str(ckpt_dir), filename="last-{epoch:02d}",
    save_last=True, every_n_epochs=1,
)
early_stop = EarlyStopping(monitor="val_acc", mode="max", patience=15, verbose=True)
lr_monitor = LearningRateMonitor(logging_interval="step")

# Mixed precision on GPU, full precision on CPU
precision = "16-mixed" if torch.cuda.is_available() else "32-true"

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator="auto", devices=1,
    precision=precision,
    logger=tb_logger,
    callbacks=[best_ckpt, last_ckpt, early_stop, lr_monitor, RichProgressBar()],
    log_every_n_steps=20, val_check_interval=1.0,
    gradient_clip_val=1.0, accumulate_grad_batches=2,
)
print("Trainer is ready | precision =", precision)


In [ ]:
# === Training ===
# From scratch:
trainer.fit(model, datamodule=dm)

# To resume (if last.ckpt exists on Drive):
# resume = ckpt_dir / "last.ckpt"
# trainer.fit(model, datamodule=dm, ckpt_path=str(resume) if resume.exists() else None)
